In [0]:
%run ../00-common/01.environment_config

In [0]:
%run ../00-common/03.Silver_Helper

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
sliver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
driver_df = spark.read.table(bronze_table).filter(col('batch_id') == v_batch_id)

In [0]:
driver_drop_df = driver_df.drop('url')

In [0]:
driver_distincts_df = driver_drop_df.dropDuplicates(["driverId"])

In [0]:
driver_renamed_df = (
    driver_distincts_df
    .withColumnsRenamed({"driverId": "driver_id","dateOfBirth":"date_of_birth"})
)

In [0]:
from pyspark.sql.functions import *

In [0]:
driver_concate_df = (
    driver_renamed_df
    .withColumn("driver_name", initcap(concat_ws(' ', col('name.givenName'), col('name.familyName'))))
    .drop('name')
)

In [0]:
driver_selects_df = (
    driver_concate_df
    .select('driver_id','driver_name','date_of_birth','nationality','ingestion_timestamp','source_files','batch_id')
    .withColumn("nationality", initcap(col('nationality')))
)

In [0]:
columns_to_update=[
        "driver_name",
        "date_of_birth",
        "nationality",
        "ingestion_timestamp",
        "source_files",
        "batch_id"
    ]

In [0]:
write_to_silver(input_df=driver_selects_df,target_table=sliver_table,
                merge_condn="t.driver_id = s.driver_id",
                columns_to_update=columns_to_update
                )

In [0]:
display(spark.read.table(sliver_table))